In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import entropy

MOVEMENT_DIR = Path(
    r"C:\Users\Sheel\OneDrive\Documents\Default Project\data\pads-parkinsons-disease-smartwatch-dataset-1.0.0\pads-parkinsons-disease-smartwatch-dataset-1.0.0\preprocessed\movement"
)

FS = 100.0

tasks = [
    "Relaxed1", "Relaxed2", "RelaxedTask1", "RelaxedTask2",
    "StretchHold", "HoldWeight", "DrinkGlas", "CrossArms",
    "TouchNose", "Entrainment1", "Entrainment2",
]
wrists = ["LeftWrist", "RightWrist"]
sensors = ["Accelerometer", "Gyroscope"]
axes = ["X", "Y", "Z"]
channels = [f"{t}_{w}_{s}_{a}" for t in tasks for w in wrists for s in sensors for a in axes]

print(len(channels), "channels")

132 channels


In [2]:
bin_files = sorted(MOVEMENT_DIR.glob("*_ml.bin"))
ids = np.array([int(f.name.split("_")[0]) for f in bin_files])
movement = np.stack([np.fromfile(f, dtype=np.float32).reshape(len(channels), -1) for f in bin_files])
print("movement:", movement.shape, movement.dtype)

movement: (469, 132, 976) float32


In [3]:
groups = {
    "Healthy": [46, 78, 81, 108, 148, 153, 160, 165, 214, 254, 295, 319, 335, 389, 463],
    "Parkinson's": [4, 95, 109, 113, 175, 206, 226, 249, 345, 347, 362, 365, 395, 406, 442],
    "Essential Tremor": [66, 70, 72, 91, 112, 124, 143, 200, 275, 278, 302, 340, 350, 386, 396],
    "Atypical Parkinsonism": [15, 18, 32, 64, 106, 110, 114, 118, 158, 168, 238, 280, 291, 304, 397],
}
group_names = list(groups)
id_to_group = {pid: g for g, pids in groups.items() for pid in pids}

mask = np.isin(ids, list(id_to_group))
mov = movement[mask]
pids = ids[mask]

channel_df = pd.DataFrame({
    "channel": channels,
    "task": [c.split("_")[0] for c in channels],
    "wrist": [c.split("_")[1] for c in channels],
    "sensor": [c.split("_")[2] for c in channels],
    "axis": [c.split("_")[3] for c in channels],
})

predictor_names = [
    "Relative Tremor Power", "Harmonic Power Ratio", "Movement Amplitude",
    "Acceleration Norm", "Jerk", "Spectral Entropy", "Primary Frequency Component",
]

def stream_predictors(acc_norm):
    n = acc_norm.shape[1]
    freqs = np.fft.rfftfreq(n, d=1 / FS)
    dfreq = FS / n
    power = np.abs(np.fft.rfft(acc_norm, axis=1)) ** 2
    power_nn = power[:, 1:]
    primary_bin = np.argmax(power_nn, axis=1) + 1
    primary_freq = freqs[primary_bin]
    spec_entropy = entropy(power_nn, axis=1)
    jerk = np.diff(acc_norm, axis=1)
    tremor_band = ((freqs >= 3) & (freqs <= 8))[1:]
    rel_tremor_power = power_nn[:, tremor_band].sum(axis=1) / power_nn.sum(axis=1)
    harm_bins = np.stack([
        np.clip(np.round(k * freqs[primary_bin] / dfreq).astype(int), 0, len(freqs) - 1)
        for k in (2, 3)
    ], axis=1)
    rows = np.arange(power.shape[0])[:, None]
    harmonic_power = power[rows, harm_bins].sum(axis=1)
    main_power = power[np.arange(power.shape[0]), primary_bin]
    return {
        "Relative Tremor Power": rel_tremor_power,
        "Harmonic Power Ratio": harmonic_power / main_power,
        "Movement Amplitude": np.ptp(acc_norm, axis=1),
        "Acceleration Norm": acc_norm.mean(axis=1),
        "Jerk": jerk.mean(axis=1),
        "Spectral Entropy": spec_entropy,
        "Primary Frequency Component": primary_freq,
    }

stream_feats = {name: [] for name in predictor_names}
for (task, wrist), grp in channel_df.groupby(["task", "wrist"]):
    acc = mov[:, grp.index[grp["sensor"] == "Accelerometer"].tolist(), :]
    acc_norm = np.linalg.norm(acc, axis=1)
    stream = stream_predictors(acc_norm)
    for name in predictor_names:
        stream_feats[name].append(stream[name])

X_model = np.column_stack([np.mean(v, axis=0) for v in stream_feats.values()])
feature_names = predictor_names
y_labels = np.array([id_to_group[p] for p in pids])
y_model = pd.Categorical(y_labels, categories=group_names).codes

print("X:", X_model.shape, "| y:", y_model.shape)
print(pd.Series(y_labels).value_counts().to_string())

X: (60, 7) | y: (60,)
Parkinson's              15
Atypical Parkinsonism    15
Healthy                  15
Essential Tremor         15


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

model_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(solver="saga", l1_ratio=0.5, max_iter=5000, random_state=0)),
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

cv_results = cross_validate(
    model_pipeline,
    X_model,
    y_model,
    cv=cv,
    return_estimator=True,
)
coefs = np.array([
    est.named_steps["clf"].coef_
    for est in cv_results["estimator"]
])
print("coefs shape:", coefs.shape)

mean_coefs = coefs.mean(axis=0)
coef_df = pd.DataFrame(
    mean_coefs,
    index=group_names,
    columns=feature_names,
)

coef_df.round(3)

coefs shape: (3, 4, 7)


,Relative Tremor Power,Harmonic Power Ratio,Movement Amplitude,Acceleration Norm,Jerk,Spectral Entropy,Primary Frequency Component
Healthy,-0.360,0.230,0.000,-0.132,0.078,0.008,-0.250
Parkinson's,-0.026,-0.024,0.351,-0.103,-0.434,-0.017,0.135
Essential Tremor,0.402,-0.071,0.181,1.467,-0.306,-0.326,0.460
Atypical Parkinsonism,-0.015,-0.046,-0.391,-0.083,0.857,1.000,-0.324


In [5]:
reference_class = "Parkinson's"

log_odds = mean_coefs - mean_coefs[group_names.index(reference_class)]
log_odds_df = pd.DataFrame(
    log_odds,
    index=group_names,
    columns=feature_names,
)

log_odds_df.round(3)

,Relative Tremor Power,Harmonic Power Ratio,Movement Amplitude,Acceleration Norm,Jerk,Spectral Entropy,Primary Frequency Component
Healthy,-0.334,0.254,-0.351,-0.028,0.512,0.025,-0.385
Parkinson's,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Essential Tremor,0.428,-0.047,-0.169,1.570,0.127,-0.309,0.325
Atypical Parkinsonism,0.011,-0.022,-0.741,0.021,1.291,1.017,-0.459
